# Predicting Students' STEM Track Choices Using Machine Learning**Journal paper analysis notebook** — Computer Applications in Engineering Education (CAEE)This notebook produces all figures (1–9) and tables (1–7) for the paper.  All plot labels are in English; the underlying dataset uses Serbian (Cyrillic) column names.

In [ ]:
# ── Google Colab: uncomment the next two lines to upload СТЕМ.xlsx ──# from google.colab import files# uploaded = files.upload()   # drag-and-drop the Excel fileimport re, warningsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy.stats import ttest_ind, chi2_contingency, wilcoxon, friedmanchisquarefrom sklearn.model_selection import (    train_test_split, StratifiedKFold, RepeatedStratifiedKFold,    GridSearchCV, RandomizedSearchCV, cross_val_score, learning_curve,)from sklearn.preprocessing import StandardScalerfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipelinefrom sklearn.feature_selection import SelectKBest, f_classiffrom sklearn.linear_model import LogisticRegressionfrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.svm import SVCfrom sklearn.ensemble import RandomForestClassifier, IsolationForestfrom sklearn.dummy import DummyClassifierfrom sklearn.metrics import (    accuracy_score, precision_score, recall_score, f1_score,    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,)from scipy.stats import loguniformtry:    from xgboost import XGBClassifierexcept ImportError:    !pip install -q xgboost    from xgboost import XGBClassifiertry:    import shapexcept ImportError:    !pip install -q shap    import shapwarnings.filterwarnings('ignore', category=FutureWarning)warnings.filterwarnings('ignore', category=UserWarning, module='shap')# ── Publication-quality plot settings ──sns.set_style("whitegrid")plt.rcParams.update({    "figure.figsize": (7, 4.5),    "figure.dpi": 150,    "savefig.dpi": 300,    "savefig.bbox": "tight",    "font.size": 10,    "axes.titlesize": 11,    "axes.labelsize": 10,    "xtick.labelsize": 9,    "ytick.labelsize": 9,    "legend.fontsize": 9,    "legend.title_fontsize": 9,})RANDOM_STATE = 42np.random.seed(RANDOM_STATE)print("All imports successful.")

---# 1. Data Loading and Quality Checks

In [ ]:
df = pd.read_excel("СТЕМ.xlsx")# Shorten long column names for subject grades/perceptionsdef rename_subject_columns(columns):    renames = {}    patterns = [        (re.compile(r"^Најчешћа оцена из наведених предмета у основној школи ми је била: (\[.+\])$"), "Најчешћа оцена"),        (re.compile(r"^Колико ти је поједини предмет био занимљив у основној школи\?\s+(\[.+\])$"), "Занимљив"),        (re.compile(r"^Колико ти је поједини предмет био тежак у основној школи\?\s+(\[.+\])$"), "Тежак"),        (re.compile(r"^Колико ти је поједини предмет био користан у основној школи\?\s+(\[.+\])$"), "Користан"),    ]    for col in columns:        if not isinstance(col, str):            continue        for pat, prefix in patterns:            m = pat.match(col)            if m:                renames[col] = f"{prefix} {m.group(1)}"                break    return renamesdf.rename(columns=rename_subject_columns(df.columns), inplace=True)print(f"Dataset shape: {df.shape}")print(f"Missing values: {df.isna().sum().sum()}")df.head(3)

## 1.1 Grade column fix & anomaly check

In [ ]:
# Fix non-integer grade value (row 775: Разред = 2.1)bad_grade = df[df['Разред:'] % 1 != 0]if len(bad_grade) > 0:    df.loc[bad_grade.index, 'Разред:'] = df.loc[bad_grade.index, 'Разред:'].round()    print(f"Fixed {len(bad_grade)} non-integer grade value(s).")# Remove one confirmed anomaly (grade = 6, outside 1-4 range for high school)out_of_range = df[~df['Разред:'].isin([1, 2, 3, 4])]print(f"Rows with grade outside 1-4: {len(out_of_range)}")

## 1.2 Target variable definition

In [ ]:
stem_tracks = [    'природно-математички',    'специјализовано одељење за ученике са посебним способностима за информатику',    'специјализовано одељење за ученике са посебним способностима за математику',    'специјализовано одељење за ученике са посебним способностима за биологију и хемију',    'специјализовано одељење за ученике са посебним способностима за физику',]df['СТЕМ'] = df['Смер:'].apply(lambda x: 1 if pd.notna(x) and x in stem_tracks else 0)n_stem = (df['СТЕМ'] == 1).sum()n_non  = (df['СТЕМ'] == 0).sum()print(f"STEM: {n_stem} ({n_stem/len(df)*100:.1f}%)")print(f"Non-STEM: {n_non} ({n_non/len(df)*100:.1f}%)")

### Figure 1 — STEM vs Non-STEM distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))counts = df['СТЕМ'].value_counts().sort_index()bars = ax.bar(['Non-STEM', 'STEM'], [counts[0], counts[1]],              color=['#5B9BD5', '#ED7D31'], edgecolor='black', width=0.5)for bar, val in zip(bars, [counts[0], counts[1]]):    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,            f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)ax.set_ylabel('Number of Students')ax.set_title('Figure 1. Distribution of STEM vs Non-STEM Students')ax.set_ylim(0, max(counts)*1.2)plt.tight_layout()plt.savefig('fig1_distribution.png')plt.show()

---# 2. Data Preprocessing

## 2.1 Drop descriptive / free-text columns

In [ ]:
cols_to_drop = [    'Timestamp',    'Занимање твоје мајке:',    'Занимање твог оца:',    'Зашто сте изабрали да упишите Гимназију?',    'Зашто сте изабрали да упишете овај смер?',]X = df.drop(columns=cols_to_drop)y = df['СТЕМ']# Convert Likert responses to numericdef likert_to_number(x):    if pd.isna(x): return np.nan    if isinstance(x, (int, np.integer)): return int(x)    if isinstance(x, float) and x == int(x): return int(x)    s = str(x).strip()    if not s: return np.nan    return int(s.split('-')[0].strip())likert_cols = [c for c in X.columns               if 'Процени' in c or c.startswith('Занимљив ')               or c.startswith('Тежак ') or c.startswith('Користан ')]for col in likert_cols:    X[col] = X[col].apply(likert_to_number)print(f"Converted {len(likert_cols)} Likert columns to numeric.")

## 2.2 Remove leaky / non-predictive columns

In [ ]:
cols_remove_model = ['СТЕМ', 'Смер:', 'Похађам средњу школу:', 'Разред:', 'Година рођења:']X = X.drop(columns=cols_remove_model)print(f"Features after removal: {X.shape[1]}")

## 2.3 Duplicate & anomaly removal

In [ ]:
# Remove exact duplicate (ignoring Timestamp)df_no_ts = df.drop(columns=['Timestamp'])dup_idx = df_no_ts[df_no_ts.duplicated(keep='first')].indexdf = df.drop(index=dup_idx)X = X.drop(index=dup_idx)y = y.drop(index=dup_idx)print(f"Removed {len(dup_idx)} duplicate row(s). Rows: {len(df)}")# Anomaly detection: Isolation Forest + straight-liningnumeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()iso = IsolationForest(contamination=0.02, random_state=RANDOM_STATE)iso_labels = pd.Series(iso.fit_predict(X[numeric_cols]), index=X.index)print(f"Isolation Forest anomalies: {(iso_labels == -1).sum()}")# Straight-lining: respondents using ≤3 distinct values across all numeric itemsn_unique = X[numeric_cols].nunique(axis=1)straight_liners = n_unique[n_unique <= 3]print(f"Straight-liners (≤3 distinct values): {len(straight_liners)}")df = df.drop(index=straight_liners.index)X = X.drop(index=straight_liners.index)y = y.drop(index=straight_liners.index)print(f"Final dataset size: {len(X)} observations")

## 2.4 Encoding categorical variables

In [ ]:
# Numeric conversion for 'number of children'children_col = 'Колико има деце у твојој породици, рачунајући и тебе (упиши број):'X[children_col] = pd.to_numeric(X[children_col], errors='coerce')X[children_col] = X[children_col].fillna(X[children_col].median())# Ordinal encodingurbanization_map = {'селу': 1, 'варошици': 2, 'мањем граду': 3, 'већем граду': 4}education_map = {'завршена основна школа': 1, 'завршена средња школа': 2,                 'завршена виша школа': 3, 'завршен факултет': 4}student_map = {'добар/а': 3, 'врло добар/а': 4, 'одличан/а': 5}X['Одрастао/ла сам у:'] = X['Одрастао/ла сам у:'].map(urbanization_map)X['Основну школу сам завршио/ла у:'] = X['Основну школу сам завршио/ла у:'].map(urbanization_map)X['Стручна спрема твоје мајке:'] = X['Стручна спрема твоје мајке:'].map(education_map)X['Стручна спрема твог оца:'] = X['Стручна спрема твог оца:'].map(education_map)X['Какав си био/ла ђак у основној школи?'] = X['Какав си био/ла ђак у основној школи?'].map(student_map)# One-hot encoding for nominal featuresnominal_cols = [    'Пол:',    'Најомиљенији предмет:',    'Други омиљени предмет (омиљени предмет, мање омиљени од претходног):',    'Трећи омиљени предмет (омиљени предмет, мање омиљени од претходна два):',]X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)print(f"Total features after encoding: {X.shape[1]}")print(f"Remaining object columns: {X.select_dtypes(include='object').columns.tolist()}")print(f"Missing values: {X.isna().sum().sum()}")

## 2.5 Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)print(f"Training set: {X_train.shape[0]} samples")print(f"Test set: {X_test.shape[0]} samples")print(f"STEM proportion — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

---# 3. ML Pipeline and Model Training

## 3.1 Pipeline construction

In [ ]:
# Identify binary (one-hot) vs continuous featuresbinary_cols = X_train.select_dtypes(include='bool').columns.tolist()numeric_cols_scale = [c for c in X_train.columns if c not in binary_cols]print(f"Continuous features (scaled): {len(numeric_cols_scale)}")print(f"Binary features (passthrough): {len(binary_cols)}")def make_pipeline(model, num_cols=None, bin_cols=None):    num_cols = numeric_cols_scale if num_cols is None else num_cols    bin_cols = binary_cols if bin_cols is None else bin_cols    preprocessor = ColumnTransformer([        ('scaler', StandardScaler(), num_cols),        ('passthrough', 'passthrough', bin_cols),    ])    return Pipeline([        ('preprocessor', preprocessor),        ('selector', SelectKBest(score_func=f_classif)),        ('model', model),    ])

## 3.2 Cross-validation setup

In [ ]:
N_SPLITS = 5skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)print(f"Stratified {N_SPLITS}-Fold Cross-Validation:")for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):    stem_prop = y_train.iloc[val_idx].mean()    print(f"  Fold {fold}: {len(tr_idx)} train / {len(val_idx)} val, STEM proportion = {stem_prop:.3f}")

## 3.3 Baseline model

In [ ]:
baseline = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)baseline.fit(X_train, y_train)y_pred_base = baseline.predict(X_test)y_proba_base = baseline.predict_proba(X_test)[:, 1]print(f"Baseline accuracy: {accuracy_score(y_test, y_pred_base):.3f}")print(f"Baseline F1 (weighted): {f1_score(y_test, y_pred_base, average='weighted'):.3f}")

## 3.4 Hyperparameter tuning — all five models

### Table 1 — Hyperparameter search spaces

In [ ]:
# ── Table 1: Hyperparameter search spaces ──table1_data = {    'Model': ['Logistic Regression', 'Logistic Regression', 'Logistic Regression',              'KNN', 'KNN', 'KNN', 'KNN',              'SVM', 'SVM', 'SVM',              'Random Forest', 'Random Forest', 'Random Forest', 'Random Forest',              'XGBoost', 'XGBoost', 'XGBoost', 'XGBoost'],    'Hyperparameter': ['C', 'l1_ratio (penalty)', 'k (features)',                       'n_neighbors', 'weights', 'metric', 'k (features)',                       'C', 'kernel', 'k (features)',                       'n_estimators', 'max_depth', 'min_samples_split', 'k (features)',                       'n_estimators', 'max_depth', 'learning_rate', 'k (features)'],    'Search Range': ['{0.01, 0.1, 1, 10, 100}', '{0 (L2), 1 (L1)}', '{10, 30, 50, 70, 90, all}',                     '{3, 5, 7, 9, 11, 15}', '{uniform, distance}', '{euclidean, manhattan}', '{10, 30, 50, 70, 90, all}',                     '{0.1, 1, 10}', '{linear, rbf}', '{10, 30, 50, 70, 90, all}',                     '{100, 200}', '{None, 10, 20}', '{2, 5}', '{10, 30, 50, 70, 90, all}',                     '{100, 200}', '{3, 5}', '{0.05, 0.1, 0.2}', '{10, 30, 50, 70, 90, all}'],}table1 = pd.DataFrame(table1_data)print("Table 1. Hyperparameter Search Spaces")print(table1.to_string(index=False))

In [ ]:
# ── Train all five models with GridSearchCV ──models_config = {    'Logistic Regression': {        'model': LogisticRegression(max_iter=1000, solver='liblinear', random_state=RANDOM_STATE),        'params': {            'model__C': [0.01, 0.1, 1, 10, 100],            'model__l1_ratio': [1, 0],            'selector__k': [10, 30, 50, 70, 90, 'all'],        },    },    'KNN': {        'model': KNeighborsClassifier(),        'params': {            'model__n_neighbors': [3, 5, 7, 9, 11, 15],            'model__weights': ['uniform', 'distance'],            'model__metric': ['euclidean', 'manhattan'],            'selector__k': [10, 30, 50, 70, 90, 'all'],        },    },    'SVM': {        'model': SVC(probability=True, random_state=RANDOM_STATE),        'params': {            'model__C': [0.1, 1, 10],            'model__kernel': ['linear', 'rbf'],            'selector__k': [10, 30, 50, 70, 90, 'all'],        },    },    'Random Forest': {        'model': RandomForestClassifier(random_state=RANDOM_STATE),        'params': {            'model__n_estimators': [100, 200],            'model__max_depth': [None, 10, 20],            'model__min_samples_split': [2, 5],            'selector__k': [10, 30, 50, 70, 90, 'all'],        },    },    'XGBoost': {        'model': XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss'),        'params': {            'model__n_estimators': [100, 200],            'model__max_depth': [3, 5],            'model__learning_rate': [0.05, 0.1, 0.2],            'selector__k': [10, 30, 50, 70, 90, 'all'],        },    },}best_estimators = {}cv_results = {}test_results = {}predictions = {}for name, cfg in models_config.items():    print(f"\n{'='*60}")    print(f"Training: {name}")    print(f"{'='*60}")    gs = GridSearchCV(        estimator=make_pipeline(cfg['model']),        param_grid=cfg['params'],        scoring='f1_weighted',        cv=skf,        n_jobs=-1,    )    gs.fit(X_train, y_train)    best = gs.best_estimator_    bi = gs.best_index_    cv_mean = gs.cv_results_['mean_test_score'][bi]    cv_std = gs.cv_results_['std_test_score'][bi]    y_pred = best.predict(X_test)    y_proba = best.predict_proba(X_test)[:, 1]    best_estimators[name] = best    cv_results[name] = (cv_mean, cv_std)    predictions[name] = {'y_pred': y_pred, 'y_proba': y_proba}    test_results[name] = {        'Accuracy': accuracy_score(y_test, y_pred),        'Precision': precision_score(y_test, y_pred, zero_division=0),        'Recall': recall_score(y_test, y_pred, zero_division=0),        'F1 (weighted)': f1_score(y_test, y_pred, average='weighted', zero_division=0),        'F1 (macro)': f1_score(y_test, y_pred, average='macro', zero_division=0),        'ROC-AUC': roc_auc_score(y_test, y_proba),    }    print(f"Best params: {gs.best_params_}")    print(f"CV F1 (weighted): {cv_mean:.3f} ± {cv_std:.3f}")    print(f"Test Accuracy: {test_results[name]['Accuracy']:.3f}")    print(f"Test ROC-AUC: {test_results[name]['ROC-AUC']:.3f}")# Also store baselinetest_results['Baseline (Dummy)'] = {    'Accuracy': accuracy_score(y_test, y_pred_base),    'Precision': precision_score(y_test, y_pred_base, zero_division=0),    'Recall': recall_score(y_test, y_pred_base, zero_division=0),    'F1 (weighted)': f1_score(y_test, y_pred_base, average='weighted', zero_division=0),    'F1 (macro)': f1_score(y_test, y_pred_base, average='macro', zero_division=0),    'ROC-AUC': roc_auc_score(y_test, y_proba_base),}predictions['Baseline (Dummy)'] = {'y_pred': y_pred_base, 'y_proba': y_proba_base}print("\n✓ All models trained successfully.")

---# 4. Results and Comparative Analysis

### Table 2 — Model performance comparison

In [ ]:
# ── Table 2: Comparative model performance ──comparison_df = pd.DataFrame(test_results).T.round(3)comparison_df.insert(0, 'CV F1 (w) ± SD', [    f"{cv_results[m][0]:.3f} ± {cv_results[m][1]:.3f}" if m in cv_results else '—'    for m in comparison_df.index])comparison_df = comparison_df.sort_values('F1 (weighted)', ascending=False)print("Table 2. Comparative Model Performance")print(comparison_df.to_string())

### Figure 4 — ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))model_order = ['Logistic Regression', 'KNN', 'SVM', 'Random Forest', 'XGBoost']colors = ['#4472C4', '#ED7D31', '#A5A5A5', '#70AD47', '#FFC000']for name, color in zip(model_order, colors):    fpr, tpr, _ = roc_curve(y_test, predictions[name]['y_proba'])    auc_val = test_results[name]['ROC-AUC']    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.3f})', color=color, linewidth=1.5)# Baselinefpr_b, tpr_b, _ = roc_curve(y_test, predictions['Baseline (Dummy)']['y_proba'])ax.plot(fpr_b, tpr_b, label='Baseline (AUC = 0.500)', color='#B4B4B4', linestyle=':', linewidth=1.2)ax.plot([0, 1], [0, 1], linestyle='--', color='grey', linewidth=0.8)ax.set_xlabel('False Positive Rate')ax.set_ylabel('True Positive Rate')ax.set_title('Figure 4. ROC Curves — All Models')ax.legend(loc='lower right', fontsize=8)plt.tight_layout()plt.savefig('fig4_roc_curves.png')plt.show()

### Figure 5 — Confusion matrix (best model)

In [ ]:
# Pick the model with highest test F1 (weighted)best_model_name = max(    [n for n in model_order],    key=lambda n: test_results[n]['F1 (weighted)'])print(f"Best model by test F1 (weighted): {best_model_name}")cm = confusion_matrix(y_test, predictions[best_model_name]['y_pred'])fig, ax = plt.subplots(figsize=(5, 4))disp = ConfusionMatrixDisplay(cm, display_labels=['Non-STEM', 'STEM'])disp.plot(ax=ax, cmap='Blues', values_format='d')ax.set_xlabel('Predicted Label')ax.set_ylabel('True Label')ax.set_title(f'Figure 5. Confusion Matrix — {best_model_name}')plt.tight_layout()plt.savefig('fig5_confusion_matrix.png')plt.show()

---# 5. Statistical Assessment of Model Performance DifferencesWe use **5×5 repeated stratified cross-validation** to generate 25 F1-weighted scores per model.Hyperparameters are tuned in an **inner CV loop** (nested CV), so the outer fold scoresare unbiased estimates used for statistical comparison.> **Note:** A simple paired t-test on CV fold results is inappropriate because> training sets overlap across folds, violating the independence assumption.

### Table 3 — Statistical comparison of model performance

In [ ]:
# ── Repeated Stratified CV (5 folds × 5 repetitions) with nested HP tuning ──rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)# Simplified param grids for inner CV to keep compute feasibleinner_grids = {    'Logistic Regression': {        'model__C': [0.1, 1, 10],        'model__l1_ratio': [0, 1],        'selector__k': [30, 70, 'all'],    },    'KNN': {        'model__n_neighbors': [5, 9, 15],        'model__weights': ['uniform', 'distance'],        'selector__k': [30, 70, 'all'],    },    'SVM': {        'model__C': [0.1, 1, 10],        'model__kernel': ['linear', 'rbf'],        'selector__k': [30, 70, 'all'],    },    'Random Forest': {        'model__n_estimators': [100, 200],        'model__max_depth': [None, 10],        'selector__k': [30, 70, 'all'],    },    'XGBoost': {        'model__n_estimators': [100, 200],        'model__max_depth': [3, 5],        'model__learning_rate': [0.05, 0.1],        'selector__k': [30, 70, 'all'],    },}print("Running 5×5 repeated stratified nested CV (this may take several minutes)...")model_scores_rcv = {}for name in model_order:    cfg = models_config[name]    nested_est = GridSearchCV(        estimator=make_pipeline(cfg['model']),        param_grid=inner_grids[name],        cv=inner_cv,        scoring='f1_weighted',        n_jobs=-1,    )    scores = cross_val_score(        nested_est, X_train, y_train, cv=rskf,        scoring='f1_weighted', n_jobs=-1,    )    model_scores_rcv[name] = scores    print(f"  {name}: {scores.mean():.3f} ± {scores.std():.3f}")print("\n✓ Repeated CV complete.")

In [ ]:
# ── Friedman test (non-parametric, for k > 2 related samples) ──scores_matrix = np.column_stack([model_scores_rcv[m] for m in model_order])stat_fried, p_fried = friedmanchisquare(*[scores_matrix[:, i] for i in range(len(model_order))])print(f"Friedman test: χ² = {stat_fried:.3f}, p = {p_fried:.4f}")# ── Pairwise Wilcoxon signed-rank tests with Bonferroni correction ──from itertools import combinationsn_comparisons = len(list(combinations(model_order, 2)))alpha_bonf = 0.05 / n_comparisonspairwise_results = []for m1, m2 in combinations(model_order, 2):    stat_w, p_w = wilcoxon(model_scores_rcv[m1], model_scores_rcv[m2])    diff = model_scores_rcv[m1].mean() - model_scores_rcv[m2].mean()    sig = 'Yes' if p_w < alpha_bonf else 'No'    pairwise_results.append({        'Model A': m1, 'Model B': m2,        'Mean Diff (A−B)': round(diff, 4),        'Wilcoxon p': round(p_w, 4),        'Significant (Bonferroni)': sig,    })table3 = pd.DataFrame(pairwise_results)print(f"\nTable 3. Pairwise Wilcoxon Signed-Rank Tests (α_Bonf = {alpha_bonf:.4f})")print(table3.to_string(index=False))

---# 6. Learning Curves### Figure 6 — F1 (weighted) vs training set size

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))train_sizes_frac = np.linspace(0.1, 1.0, 10)colors_lc = {'Logistic Regression': '#4472C4', 'KNN': '#ED7D31',             'SVM': '#A5A5A5', 'Random Forest': '#70AD47', 'XGBoost': '#FFC000'}for name in model_order:    est = best_estimators[name]    train_sizes_abs, train_scores, val_scores = learning_curve(        est, X_train, y_train,        train_sizes=train_sizes_frac,        cv=skf,        scoring='f1_weighted',        n_jobs=-1,        random_state=RANDOM_STATE,    )    val_mean = val_scores.mean(axis=1)    val_std = val_scores.std(axis=1)    ax.plot(train_sizes_abs, val_mean, 'o-', label=name,            color=colors_lc[name], linewidth=1.5, markersize=4)    ax.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std,                    alpha=0.15, color=colors_lc[name])ax.set_xlabel('Training Set Size')ax.set_ylabel('F1 Score (weighted)')ax.set_title('Figure 6. Learning Curves — Validation F1 vs Training Size')ax.legend(fontsize=8)ax.set_ylim(0.5, 0.85)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('fig6_learning_curves.png')plt.show()

---# 7. Model Interpretability

## 7.1 Logistic Regression coefficients and odds ratios

### Table 5 — LR coefficients and odds ratios

In [ ]:
def get_selected_features(pipe):    names = pipe.named_steps['preprocessor'].get_feature_names_out()    mask = pipe.named_steps['selector'].get_support()    return [n.split('__', 1)[1] for n, keep in zip(names, mask) if keep]# ── English feature-name mapping ──FEAT_EN = {    'Занимљив [Физика]': 'Physics interestingness',    'Занимљив [Математика]': 'Math interestingness',    'Тежак [Физика]': 'Physics difficulty',    'Тежак [Математика]': 'Math difficulty',    'Користан [Физика]': 'Physics usefulness',    'Користан [Математика]': 'Math usefulness',    'Најчешћа оцена [Математика]': 'Math grade',    'Најчешћа оцена [Физика]': 'Physics grade',    'Најчешћа оцена [Информатика]': 'Informatics grade',    'Најчешћа оцена [Хемија]': 'Chemistry grade',    'Најчешћа оцена [Биологија]': 'Biology grade',    'Најчешћа оцена [Српски језик и књижевност]': 'Serbian lang. grade',    'Пол:_мушки': 'Gender (male)',    'Одрастао/ла сам у:': 'Grew up in (urbanization)',    'Основну школу сам завршио/ла у:': 'Primary school location',    'Стручна спрема твоје мајке:': "Mother's education",    'Стручна спрема твог оца:': "Father's education",    'Какав си био/ла ђак у основној школи?': 'Academic achievement',    'Колико има деце у твојој породици, рачунајући и тебе (упиши број):': 'Number of children',}def translate_feature(name):    """Translate Serbian feature name to English, with fallback."""    if name in FEAT_EN:        return FEAT_EN[name]    # Handle one-hot encoded features    for prefix_sr, prefix_en in [        ('Најомиљенији предмет:_', 'Fav. subject: '),        ('Други омиљени предмет', '2nd fav. subject: '),        ('Трећи омиљени предмет', '3rd fav. subject: '),        ('Процени који од фактора је имао колико утицаја', 'Enrollment factor: '),    ]:        if name.startswith(prefix_sr):            suffix = name[len(prefix_sr):].strip('_ ')            return prefix_en + suffix[:30]    # Likert items    for pat_sr, pat_en in [        ('Занимљив [', 'Interesting ['), ('Тежак [', 'Difficult ['),        ('Користан [', 'Useful ['), ('Најчешћа оцена [', 'Grade ['),    ]:        if name.startswith(pat_sr):            return name.replace(pat_sr, pat_en)    # Enrollment factors    if 'Процени' in name or 'утицаја' in name:        return 'Enrollment factor'    return name[:45]lr_best = best_estimators['Logistic Regression']coef_df = pd.DataFrame({    'Feature (Serbian)': get_selected_features(lr_best),    'Coefficient': lr_best.named_steps['model'].coef_[0],})coef_df['Feature'] = coef_df['Feature (Serbian)'].apply(translate_feature)coef_df['Odds Ratio'] = np.exp(coef_df['Coefficient'])coef_df['|Coefficient|'] = coef_df['Coefficient'].abs()n_zero = (coef_df['Coefficient'] == 0).sum()print(f"Selected features (k): {len(coef_df)}")print(f"Coefficients zeroed out by L1: {n_zero} / {len(coef_df)}")top15_lr = coef_df.sort_values('|Coefficient|', ascending=False).head(15)print("\nTable 5. Top 15 Logistic Regression Coefficients and Odds Ratios")print(top15_lr[['Feature', 'Coefficient', 'Odds Ratio']].to_string(index=False))

## 7.2 Feature importance (Random Forest & XGBoost)

### Figure 9 — Comparative feature importance

In [ ]:
rf_best_est = best_estimators['Random Forest']xgb_best_est = best_estimators['XGBoost']rf_imp = pd.DataFrame({    'Feature (Serbian)': get_selected_features(rf_best_est),    'Importance_RF': rf_best_est.named_steps['model'].feature_importances_,})xgb_imp = pd.DataFrame({    'Feature (Serbian)': get_selected_features(xgb_best_est),    'Importance_XGB': xgb_best_est.named_steps['model'].feature_importances_,})combined_imp = rf_imp.merge(xgb_imp, on='Feature (Serbian)')combined_imp['Feature'] = combined_imp['Feature (Serbian)'].apply(translate_feature)combined_imp['Avg Rank'] = (    combined_imp['Importance_RF'].rank(ascending=False) +    combined_imp['Importance_XGB'].rank(ascending=False)) / 2top15_fi = combined_imp.sort_values('Avg Rank').head(15)# Plotfig, ax = plt.subplots(figsize=(7.5, 5.5))plot_melt = top15_fi.melt(    id_vars='Feature', value_vars=['Importance_RF', 'Importance_XGB'],    var_name='Model', value_name='Importance',)plot_melt['Model'] = plot_melt['Model'].map({    'Importance_RF': 'Random Forest', 'Importance_XGB': 'XGBoost',})sns.barplot(data=plot_melt, x='Importance', y='Feature', hue='Model', ax=ax,            palette={'Random Forest': '#70AD47', 'XGBoost': '#FFC000'})ax.set_xlabel('Feature Importance')ax.set_ylabel('')ax.set_title('Figure 9. Feature Importance — Random Forest vs XGBoost (Top 15)')ax.legend(title='')plt.tight_layout()plt.savefig('fig9_feature_importance.png')plt.show()

## 7.3 SHAP analysis

### Figure 7 — SHAP beeswarm summary plot

In [ ]:
# SHAP TreeExplainer on Random Forestfeat_rf = get_selected_features(rf_best_est)feat_rf_en = [translate_feature(f) for f in feat_rf]X_test_trans = rf_best_est[:-1].transform(X_test)explainer = shap.TreeExplainer(rf_best_est.named_steps['model'])shap_values = explainer.shap_values(X_test_trans)# shap_values shape: (n_samples, n_features, n_classes) or list of 2 arraysif isinstance(shap_values, list):    shap_values_stem = shap_values[1]elif shap_values.ndim == 3:    shap_values_stem = shap_values[:, :, 1]else:    shap_values_stem = shap_values# Beeswarm summary plot (MOST IMPORTANT FIGURE)fig = plt.figure(figsize=(7.5, 6))shap.summary_plot(shap_values_stem, X_test_trans,                  feature_names=feat_rf_en, show=False, max_display=15)plt.title('Figure 7. SHAP Summary Plot (Beeswarm) — STEM Prediction', fontsize=10, pad=10)plt.tight_layout()plt.savefig('fig7_shap_beeswarm.png')plt.show()

### Figure 8 — SHAP bar plot (mean |SHAP|)

In [ ]:
mean_abs_shap = np.abs(shap_values_stem).mean(axis=0)shap_imp_df = pd.DataFrame({    'Feature': feat_rf_en,    'Mean |SHAP|': mean_abs_shap,}).sort_values('Mean |SHAP|', ascending=False)top15_shap = shap_imp_df.head(15)fig, ax = plt.subplots(figsize=(7, 5.5))sns.barplot(data=top15_shap, x='Mean |SHAP|', y='Feature', ax=ax, color='#5B9BD5')ax.set_xlabel('Mean |SHAP Value|')ax.set_ylabel('')ax.set_title('Figure 8. Mean Absolute SHAP Contributions (Top 15)')plt.tight_layout()plt.savefig('fig8_shap_bar.png')plt.show()

### Table 4 — Top predictors across three interpretation methods

In [ ]:
# Rank synthesis across LR coefficients, ensemble importance, and SHAPrank_lr = coef_df[['Feature (Serbian)', '|Coefficient|']].copy()rank_lr['Rank_LR'] = rank_lr['|Coefficient|'].rank(ascending=False)rank_fi = combined_imp[['Feature (Serbian)', 'Avg Rank']].rename(columns={'Avg Rank': 'Rank_FI'})shap_imp_sr = pd.DataFrame({    'Feature (Serbian)': feat_rf,    'Mean |SHAP|': mean_abs_shap,})shap_imp_sr['Rank_SHAP'] = shap_imp_sr['Mean |SHAP|'].rank(ascending=False)synthesis = (    rank_lr[['Feature (Serbian)', 'Rank_LR']]    .merge(rank_fi, on='Feature (Serbian)')    .merge(shap_imp_sr[['Feature (Serbian)', 'Rank_SHAP']], on='Feature (Serbian)'))synthesis['Avg Rank'] = synthesis[['Rank_LR', 'Rank_FI', 'Rank_SHAP']].mean(axis=1)synthesis['Feature'] = synthesis['Feature (Serbian)'].apply(translate_feature)synthesis = synthesis.sort_values('Avg Rank')print("Table 4. Top 15 Predictors — Synthetic Ranking Across Three Methods")print(synthesis[['Feature', 'Rank_LR', 'Rank_FI', 'Rank_SHAP', 'Avg Rank']].head(15).to_string(index=False))

---# 8. Fairness and Robustness Analysis

### Table 6 — Performance by gender

In [ ]:
male_mask = X_test['Пол:_мушки'].values.astype(bool)female_mask = ~male_maskdef baseline_accuracy(y):    p = y.mean()    return max(p, 1 - p)def group_metrics(y_true, y_pred, y_proba):    acc = accuracy_score(y_true, y_pred)    return {        'Accuracy': acc,        'Above Baseline': acc - baseline_accuracy(y_true),        'F1 (weighted)': f1_score(y_true, y_pred, average='weighted'),        'ROC-AUC': roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,    }fairness_results = {}for name in model_order:    model = best_estimators[name]    for gender, mask in [('Male', male_mask), ('Female', female_mask)]:        y_sub = y_test[mask]        y_pred = model.predict(X_test[mask])        y_proba = model.predict_proba(X_test[mask])[:, 1]        fairness_results[(name, gender)] = group_metrics(y_sub, y_pred, y_proba)table6 = pd.DataFrame(fairness_results).T.round(3)table6.index.names = ['Model', 'Gender']print("Table 6. Model Performance by Gender")print(table6)# Gender group sizesfor g, m in [('Male', male_mask), ('Female', female_mask)]:    print(f"\n{g}: n={m.sum()}, STEM proportion={y_test[m].mean():.3f}")

### Bias analysis (gender & urbanization)

In [ ]:
rural_mask = (X_test['Одрастао/ла сам у:'] == 1).valuesurban_mask = ~rural_maskdef bias_metrics(model, X_te, y_true, mask):    y_sub = y_true[mask].values    y_pred = model.predict(X_te[mask])    pos_rate = y_pred.mean()    true_neg = y_sub == 0    true_pos = y_sub == 1    fpr = (y_pred[true_neg] == 1).mean() if true_neg.sum() > 0 else np.nan    fnr = (y_pred[true_pos] == 0).mean() if true_pos.sum() > 0 else np.nan    return {'Positive Pred. Rate': pos_rate, 'FPR': fpr, 'FNR': fnr, 'n': int(mask.sum())}rf_model = best_estimators['Random Forest']groups = {    'Male': male_mask, 'Female': female_mask,    'Rural': rural_mask, 'Urban': urban_mask,}bias_res = {name: bias_metrics(rf_model, X_test, y_test, mask) for name, mask in groups.items()}bias_table = pd.DataFrame(bias_res).T.round(3)print("Bias Metrics (Random Forest) by Group:")print(bias_table)

### Table 7 — Robustness analysis

In [ ]:
# ── Table 7: Robustness — removing key predictors ──grid_rf_robust = {    'model__n_estimators': [100, 200],    'model__max_depth': [None, 10, 20],    'model__min_samples_split': [2, 5],    'selector__k': [10, 30, 50, 70, 90, 'all'],}def rf_without_columns(cols_to_drop):    Xtr = X_train.drop(columns=cols_to_drop, errors='ignore')    Xte = X_test.drop(columns=cols_to_drop, errors='ignore')    num = [c for c in numeric_cols_scale if c in Xtr.columns]    bincols = [c for c in binary_cols if c in Xtr.columns]    gs = GridSearchCV(        make_pipeline(RandomForestClassifier(random_state=RANDOM_STATE), num_cols=num, bin_cols=bincols),        grid_rf_robust, scoring='f1_weighted', cv=skf, n_jobs=-1,    )    gs.fit(Xtr, y_train)    est = gs.best_estimator_    yp = est.predict(Xte)    ypr = est.predict_proba(Xte)[:, 1]    return {        'Accuracy': accuracy_score(y_test, yp),        'F1 (weighted)': f1_score(y_test, yp, average='weighted'),        'ROC-AUC': roc_auc_score(y_test, ypr),    }# Full model resultsrf_full = {    'Accuracy': test_results['Random Forest']['Accuracy'],    'F1 (weighted)': test_results['Random Forest']['F1 (weighted)'],    'ROC-AUC': test_results['Random Forest']['ROC-AUC'],}# (a) Remove single top predictortop_predictor = 'Занимљив [Физика]'rf_no_top = rf_without_columns([top_predictor])# (b) Remove entire "enrollment decision factors" blockenrollment_factors = [c for c in X_train.columns                      if c.startswith('Процени који од фактора је имао колико утицаја')]print(f"Enrollment decision factors block: {len(enrollment_factors)} columns")rf_no_block = rf_without_columns(enrollment_factors)table7 = pd.DataFrame({    'All features': rf_full,    'Without top predictor (Physics interest.)': rf_no_top,    f'Without enrollment factors ({len(enrollment_factors)} cols)': rf_no_block,}).T.round(3)print("\nTable 7. Robustness Analysis — Random Forest")print(table7)

---# 9. SummaryAll figures and tables have been generated with English labels.**Figures saved:**- `fig1_distribution.png` — STEM vs Non-STEM distribution- `fig4_roc_curves.png` — ROC curves for all models- `fig5_confusion_matrix.png` — Confusion matrix (best model)- `fig6_learning_curves.png` — Learning curves (F1 vs training size)- `fig7_shap_beeswarm.png` — SHAP beeswarm summary plot- `fig8_shap_bar.png` — Mean |SHAP| bar plot (top 15)- `fig9_feature_importance.png` — RF vs XGBoost feature importance*Figures 2 (preprocessing flow) and 3 (pipeline diagram) are schematic diagrams created separately.*

In [ ]:
print("\n" + "="*60)print("NOTEBOOK EXECUTION COMPLETE")print("="*60)print(f"\nDataset: {len(X)} observations, {X.shape[1]} features")print(f"Train/Test split: {len(X_train)}/{len(X_test)}")print(f"\nBest model by test F1: {best_model_name}")print(f"  Accuracy: {test_results[best_model_name]['Accuracy']:.3f}")print(f"  ROC-AUC: {test_results[best_model_name]['ROC-AUC']:.3f}")print(f"\nAll figures saved as PNG files.")